# `agg` — Reference

`agg` aggregates numeric columns with grouping inherited from standalone `pt.by()` (or computes a whole-table summary when no grouping is set).

### Primary Calling Paradigms

1. **Explicit Aliasing (`alias = "col:operation"`) — Recommended**:
   Every output column is explicitly declared as a keyword argument. No columns are created on the fly with unexpected suffixes.
   ```python
   df.by("species").agg(
       n="count",
       avg_bill="bill_length_mm:mean",
       total_mass="body_mass_g:sum",
       max_mass="body_mass_g:max",
   )
   ```

2. **1-to-1 Column Preservation (`col = "operation"`)**:
   Keep the original column name unchanged when performing a single aggregation on that column:
   ```python
   df.by("species").agg(
       n="count",
       bill_length_mm="mean",
       body_mass_g="sum",
   )
   ```

3. **Mixing Both Paradigms**:
   Combine preserved column names and explicit aliases seamlessly in a single `.agg()` call:
   ```python
   tips.by("smoker", "time").agg(
       cnt="count",
       total_bill="mean",       # preserved 1-to-1 column name
       avg_tip="tip:mean",      # explicit alias
       max_tip="tip:max",       # explicit alias
   )
   ```

4. **Columns with Spaces (String Specifications)**:
   Bracket spaced columns matching the CLI `-agg` flag:
   ```python
   df.by("species").agg("[avg bill] = [total bill]:mean, cnt = count")
   ```

5. **Table-Wide Uniform Aggregations**:
   Aggregate all numeric columns uniformly across groups:
   ```python
   df.by("species").agg(a=["mean", "count"])
   ```

6. **Whole-Table Summary**:
   Omit `.by(...)` to compute grand totals across the entire table:
   ```python
   df.agg(n="count", avg_bill="bill_length_mm:mean")
   ```

---


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
tips = pt.sample_data['tips']
titanic = pt.sample_data['titanic']


## Explicit Aliasing (`alias = "col:operation"`) — Recommended

Every output column name is explicitly declared on the left side of `=`. No columns are created on the fly, eliminating any guessing or magical suffixes.


In [2]:
# Explicit aliases: every output column is named directly (no on-the-fly column creation)
(penguins
 .by('species')
 .agg(
     n='count',
     bill_length_mm='mean',
     mass_sum='body_mass_g:sum',
     mass_mean='body_mass_g:mean',
 )
)


species,n,bill_length_mm,mass_sum,mass_mean
str,i64,f64,f64,f64
"""Adelie""",152,38.791391,558800.0,3700.662252
"""Chinstrap""",68,48.833824,253850.0,3733.088235
"""Gentoo""",124,47.504878,624350.0,5076.01626


## Mixing Preserved Column Names & Explicit Aliases

You can cleanly combine 1-to-1 preserved column names (`col='operation'`) and explicit aliases (`alias='col:operation'`) in the same `.agg()` call. Output column order strictly follows parameter order.


In [3]:
# Place count first (n at front), then preserved columns and explicit aliases:
# - cnt='count': row count
# - total_bill='mean': 1-to-1 preserved column name (mean of total_bill kept as total_bill)
# - avg_tip='tip:mean', max_tip='tip:max': explicit aliases (no columns created on the fly)
(
    tips
    .select('smoker', 'time', 'total_bill', 'tip')
    .by('smoker', 'time')
    .agg(
        cnt='count',
        total_bill='mean',
        avg_tip='tip:mean',
        max_tip='tip:max',
    )
)


smoker,time,cnt,total_bill,avg_tip,max_tip
cat,cat,i64,f64,f64,f64
"""No""","""Dinner""",106,20.09566,3.126887,9.0
"""No""","""Lunch""",45,17.050889,2.673778,6.7
"""Yes""","""Dinner""",70,21.859429,3.066,10.0
"""Yes""","""Lunch""",23,17.39913,2.834348,5.0


In [4]:
# You can name the count column anything you like as a keyword argument
# Here maddy='count' creates a column called 'maddy' that holds the group count
(
    penguins
    .select('species', 'island', 'body_mass_g')
    .by('species')
    .agg(maddy='count', body_mass_g='sum')
)


species,maddy,body_mass_g
str,i64,f64
"""Adelie""",152,558800.0
"""Chinstrap""",68,253850.0
"""Gentoo""",124,624350.0


In [5]:
# Multiple count columns as keyword arguments — each count gets its own name
(
    penguins
    .select('species', 'island', 'body_mass_g')
    .by('species')
    .agg(count='count', countv2='count', body_mass_g='mean')
)


species,count,countv2,body_mass_g
str,i64,i64,f64
"""Adelie""",152,152,3700.662252
"""Chinstrap""",68,68,3733.088235
"""Gentoo""",124,124,5076.01626


## Table-Wide Uniform Aggregations (`a=...`)

When you want to apply the same aggregation function(s) across **all** numeric columns uniformly without declaring each column individually, pass `a` or a list. Regardless of list order, `n` is always placed first.


In [6]:
# mean + n: regardless of list order, n always comes first before aggregated columns
(penguins
 .select('species', 'island', 'bill_length_mm', 'body_mass_g')
 .by('species', 'island')
 .agg(a=['mean', 'n'])  # notice n comes first
)


species,island,n,bill_length_mm,body_mass_g
str,str,i64,f64,f64
"""Adelie""","""Biscoe""",44,38.975,3709.659091
"""Adelie""","""Dream""",56,38.501786,3688.392857
"""Adelie""","""Torgersen""",52,38.95098,3706.372549
"""Chinstrap""","""Dream""",68,48.833824,3733.088235
"""Gentoo""","""Biscoe""",124,47.504878,5076.01626


In [7]:
# Multiple numeric aggregations across all numeric columns
(penguins
 .select('species', 'bill_length_mm', 'body_mass_g')
 .by('species')
 .agg(a=['sum', 'mean', 'max', 'n'])
)


species,n,bill_length_mm_sum,bill_length_mm_mean,bill_length_mm_max,body_mass_g_sum,body_mass_g_mean,body_mass_g_max
str,i64,f64,f64,f64,f64,f64,f64
"""Adelie""",152,5857.5,38.791391,46.0,558800.0,3700.662252,4775.0
"""Chinstrap""",68,3320.7,48.833824,58.0,253850.0,3733.088235,4800.0
"""Gentoo""",124,5843.1,47.504878,59.6,624350.0,5076.01626,6300.0


In [8]:
# median is a standard aggfunc — works across all numeric columns or as a keyword argument
(
    penguins
    .select('species', 'island', 'bill_length_mm', 'body_mass_g')
    .by('species')
    .agg(a=['median', 'mean', 'n'])
)


species,n,bill_length_mm_median,bill_length_mm_mean,body_mass_g_median,body_mass_g_mean
str,i64,f64,f64,f64,f64
"""Adelie""",152,38.8,38.791391,3700.0,3700.662252
"""Chinstrap""",68,49.55,48.833824,3700.0,3733.088235
"""Gentoo""",124,47.3,47.504878,5000.0,5076.01626


## Handling Missing Group Keys (`dropna`)


In [9]:
# Default dropna=False keeps missing grouping keys in their own group (no silent data loss)
(penguins
 .select('species', 'sex', 'bill_length_mm', 'body_mass_g')
 .by('species', 'sex')
 .agg(a=['mean', 'n'])
)


species,sex,n,bill_length_mm,body_mass_g
str,str,i64,f64,f64
"""Adelie""","""Female""",73,37.257534,3368.835616
"""Adelie""","""Male""",73,40.390411,4043.493151
"""Adelie""",null,6,37.84,3540.0
"""Chinstrap""","""Female""",34,46.573529,3527.205882
"""Chinstrap""","""Male""",34,51.094118,3938.970588
"""Gentoo""","""Female""",58,45.563793,4679.741379
"""Gentoo""","""Male""",61,49.47377,5484.836066
"""Gentoo""",null,5,45.625,4587.5


In [10]:
# Pass dropna=True to exclude groups with NaN keys
(penguins
 .select('species', 'sex', 'bill_length_mm', 'body_mass_g')
 .by('species', 'sex')
 .agg(a=['mean', 'n'], dropna=True)
)


species,sex,n,bill_length_mm,body_mass_g
str,str,i64,f64,f64
"""Adelie""","""Female""",73,37.257534,3368.835616
"""Adelie""","""Male""",73,40.390411,4043.493151
"""Chinstrap""","""Female""",34,46.573529,3527.205882
"""Chinstrap""","""Male""",34,51.094118,3938.970588
"""Gentoo""","""Female""",58,45.563793,4679.741379
"""Gentoo""","""Male""",61,49.47377,5484.836066


## `n`-only Aggregation — Count With No Numeric Columns Needed


In [11]:
# Count only — useful for frequency tables on categorical columns
(penguins
 .select(exclude_dtype='number')
 .by('species', 'island')
 .agg(a='count')
)


species,island,n
str,str,i64
"""Adelie""","""Biscoe""",44
"""Adelie""","""Dream""",56
"""Adelie""","""Torgersen""",52
"""Chinstrap""","""Dream""",68
"""Gentoo""","""Biscoe""",124


## Pipeline Chaining: Filter -> Select -> Aggregation -> Arrange

Because `pytae` operations return native `DataFrame` objects, you can chain verbs seamlessly end-to-end:


In [12]:
(
    penguins
    .filter('body_mass_g > 3000')
    .select('species', 'sex', 'body_mass_g')
    .by('species', 'sex')
    .agg(
        cnt='count',
        avg_mass='body_mass_g:mean',
        max_mass='body_mass_g:max',
    )
    .arrange('avg_mass desc')
)


species,sex,cnt,avg_mass,max_mass
str,str,i64,f64,f64
"""Gentoo""","""Male""",61,5484.836066,6300.0
"""Gentoo""","""Female""",58,4679.741379,5200.0
"""Gentoo""",null,4,4587.5,4875.0
"""Adelie""","""Male""",73,4043.493151,4775.0
"""Chinstrap""","""Male""",34,3938.970588,4800.0
"""Adelie""",null,4,3681.25,4250.0
"""Chinstrap""","""Female""",32,3572.65625,4150.0
"""Adelie""","""Female""",65,3424.615385,3900.0


## Columns with Spaces & String Specifications

When column names have spaces, passing them as keyword arguments (`[total bill]='mean'`) is forbidden by Python syntax.

`agg` solves this by accepting **string mapping specifications** (matching the CLI `-agg` flag), where column names with spaces are enclosed in square brackets `[col]`. No dictionary unpacking `**{...}` is required:


In [13]:
# Create DataFrame with spaced column names
tips_spaced = tips.rename(columns={'total_bill': 'total bill'})

# Single string mapping with bracketed spaced columns
(
    tips_spaced
    .by('smoker')
    .agg("tip = mean, [total bill] = mean, n = count")
)


smoker,tip,total bill,n
cat,f64,f64,i64
"""No""",2.991854,19.188278,151
"""Yes""",3.00871,20.756344,93


In [14]:
# Multiple positional string specifications
(
    tips_spaced
    .by('smoker')
    .agg("tip = mean", "[total bill] = mean", "n = count")
)


smoker,tip,total bill,n
cat,f64,f64,i64
"""No""",2.991854,19.188278,151
"""Yes""",3.00871,20.756344,93


In [15]:
# Custom output names and bracketed source column
(
    tips_spaced
    .by('smoker')
    .agg("[avg bill] = [total bill]:mean, [total tip] = tip:sum, [table count] = count")
)


smoker,avg bill,total tip,table count
cat,f64,f64,i64
"""No""",19.188278,451.77,151
"""Yes""",20.756344,279.81,93


In [16]:
# Complete pipeline: filter -> select -> agg with bracketed spaced columns across every verb
(
    tips
    .rename(columns={'total_bill': 'total bill'})
    .filter("[total bill] > 10", day=['Sat', 'Sun'], time='Dinner', size='>= 2')
    .select('smoker', 'tip', '[total bill]')
    .by('smoker')
    .agg("tip = mean, [total bill] = mean, n = count")
)


smoker,tip,total bill,n
cat,f64,f64,i64
"""No""",3.225464,20.705876,97
"""Yes""",3.087719,23.232281,57
